# Lab 2 — Consistency & Stability

**Goal:** Measure how consistent your agent is. The same question should get the same answer.

**Why:** Inconsistency erodes user trust. If a user asks the same question twice and gets contradictory answers, they lose confidence in the system. Temperature, sampling, and prompt phrasing all affect consistency.

**Metrics:**
- Length variance (is the answer always a similar length?)
- Semantic similarity (do different runs say the same thing?)
- Factual consistency (do key facts stay the same across runs?)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from eval_harness import consistency_score
from openai import OpenAI

client = OpenAI()
print('Ready ✓')

## 1. High-temperature agent (inconsistent)

In [ ]:
def high_temp_agent(prompt: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=1.5,
        max_tokens=200,
    )
    return resp.choices[0].message.content

def low_temp_agent(prompt: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=200,
    )
    return resp.choices[0].message.content


PROMPT = 'What are the three main benefits of containerisation with Docker?'

print('Running consistency check (5 runs each)...')
high_stats = consistency_score(high_temp_agent, PROMPT, n=5)
low_stats  = consistency_score(low_temp_agent, PROMPT, n=5)

print('\nHigh temperature (1.5):')
print(f'  Length:      mean={high_stats["mean_length"]:.0f}  std={high_stats["std_length"]:.0f}')
print(f'  Semantic sim: {high_stats["semantic_similarity"]:.2f}')

print('\nLow temperature (0.0):')
print(f'  Length:      mean={low_stats["mean_length"]:.0f}  std={low_stats["std_length"]:.0f}')
print(f'  Semantic sim: {low_stats["semantic_similarity"]:.2f}')

## 2. Compare outputs side by side

In [ ]:
print('=== High temperature outputs ===')
for i, output in enumerate(high_stats['outputs']):
    print(f'Run {i+1}: {output[:200]}')
    print()

print('=== Low temperature outputs ===')
for i, output in enumerate(low_stats['outputs']):
    print(f'Run {i+1}: {output[:200]}')
    print()

## 3. Factual consistency check
For factual questions, check if key facts stay the same across runs.

In [ ]:
FACTUAL_PROMPT = 'What year was Python first released, and who created it?'

# Run 5 times and extract key facts
factual_outputs = []
for _ in range(5):
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': FACTUAL_PROMPT}],
        temperature=0.5,
        max_tokens=100,
    )
    factual_outputs.append(resp.choices[0].message.content)

# Check for consistent facts
facts_to_check = ['1991', 'Guido', 'van Rossum']
print('Fact consistency across 5 runs:')
for fact in facts_to_check:
    mentions = sum(fact.lower() in o.lower() for o in factual_outputs)
    print(f'  "{fact}": mentioned in {mentions}/5 runs ({mentions/5:.0%})')

## 4. Set a consistency threshold for CI

In [ ]:
CONSISTENCY_THRESHOLD = 0.85  # semantic similarity must be >= this to pass

test_cases = [
    ('What is 2+2?', True),   # should be highly consistent
    ('Write a haiku about autumn.', False),  # should be inconsistent (creative)
]

for prompt, expect_consistent in test_cases:
    stats = consistency_score(low_temp_agent, prompt, n=3)
    is_consistent = stats['semantic_similarity'] >= CONSISTENCY_THRESHOLD
    status = '✅' if is_consistent == expect_consistent else '⚠️'
    print(f'{status} "{prompt[:40]}..."')
    print(f'   Similarity: {stats["semantic_similarity"]:.2f} (threshold: {CONSISTENCY_THRESHOLD})')
    print(f'   Consistent: {is_consistent} (expected: {expect_consistent})')

## Exercise
Build a `ConsistencyRegression` class that:
1. Runs a set of prompts at eval time and stores their consistency scores as a baseline
2. On subsequent runs, compares new scores to the baseline
3. Flags a regression if any score drops by more than 10 percentage points

This is the kind of check you'd run in CI before deploying a new model or system prompt.

In [ ]:
# Your code here
